# 第108章 银行营销响应预测项目

使用 UCI Bank Marketing 公开数据，按照二分类教学流程建立客户响应预测模型，重点学习事后泄漏、类别不平衡和阈值评价。

## 项目背景

目标是在通话开始前预测客户是否可能认购定期存款。模型学习历史响应关系，不回答一次电话是否会对特定客户产生因果增量。

## 学习目标

- 审计重复、unknown和正类比例
- 识别并排除duration事后泄漏
- 使用分层训练验证测试划分
- 建立混合类型预处理Pipeline
- 比较Dummy、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片和特征重要性理解模型


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| age/job/education | 客户画像 | 数值与类别特征 |
| duration | 本次通话时长 | 通话结束后才知道，禁止使用 |
| campaign/pdays/previous | 接触历史 | 活动相关特征 |
| poutcome | 上次活动结果 | 历史信号 |
| y | 是否认购 | 二分类目标 |

## 数据质量检查清单

- 分号分隔及字段类型
- unknown的数量和含义
- 重复记录与campaign长尾
- 正类比例和多数类准确率
- duration事后泄漏
- 三组数据的类别比例


## 项目任务

1. 明确通话前预测时点与目标
2. 审计数据质量和类别不平衡
3. 清理重复并探索响应差异
4. 识别duration等禁止字段
5. 分层划分训练、验证和测试
6. 建立混合类型预处理Pipeline
7. 比较Dummy、逻辑回归和随机森林
8. 评价PR-AUC、阈值、Lift与覆盖率
9. 分析错误类型和客户分组
10. 解释特征重要性并总结模型局限


## 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 原始数据质量与类别不平衡审计

检查分隔符、重复、unknown、目标比例和长尾变量。


In [ ]:
import numpy as np
import pandas as pd
from js import window
raw=pd.read_csv(f"{window.location.origin}/datasets/bank_marketing_full.csv",sep=';')
unknown=(raw.astype(str)=='unknown').sum().sort_values(ascending=False)
audit=pd.Series({'行数':len(raw),'重复':raw.duplicated().sum(),'正类率':(raw.y=='yes').mean(),'多数类准确率':max((raw.y=='yes').mean(),(raw.y!='yes').mean()),'campaign_P99':raw.campaign.quantile(.99)})
print(audit.round(3).to_string()); print('unknown最多字段:\n',unknown.head(8))


## 2. 清理重复并探索响应差异

unknown保留为显式类别，因为未知并不等于否；描述性组间差异不代表营销因果效果。


In [ ]:
df=raw.drop_duplicates().copy(); df['target']=(df.y=='yes').astype(int)
job_response=df.groupby('job').target.agg(['size','mean']).query('size>=200').sort_values('mean',ascending=False)
contact_response=df.groupby('contact').target.agg(['size','mean']).sort_values('mean',ascending=False)
print('清理后:',len(df),'正类率:',f'{df.target.mean():.2%}'); display(job_response.round(3)); display(contact_response.round(3))


## 3. 定义预测时点并检查泄漏

duration只有通话结束后才能获得，在通话前响应预测中属于典型事后泄漏。


In [ ]:
forbidden=['y','target','duration']; features=[column for column in df.columns if column not in forbidden]
assert not set(features)&set(forbidden)
print('预测时点: 通话开始前'); print('禁止字段:',forbidden); print('可用特征数量:',len(features)); print('duration与目标的组间均值仅用于说明泄漏风险:\n',df.groupby('target').duration.mean().round(1))


## 4. 分层划分与Dummy基线

分层划分保持三组正类比例一致；验证集选模型，测试集只用于最终评价。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score
X=df[features]; y=df.target; X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=108); X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.2,stratify=y_dev,random_state=108)
dummy=DummyClassifier(strategy='prior').fit(X_train,y_train); dummy_probability=dummy.predict_proba(X_val)[:,1]
print('训练/验证/测试:',len(X_train),len(X_val),len(X_test)); print('正类率:',*[round(part.mean(),3) for part in [y_train,y_val,y_test]]); print('Dummy PR-AUC:',round(average_precision_score(y_val,dummy_probability),3))


## 5. 建立混合类型预处理Pipeline

类别变量独热编码、数值变量标准化，并把预处理与模型绑定，避免数据处理泄漏。


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
cat=X_train.select_dtypes('object').columns.tolist(); num=[column for column in features if column not in cat]
preprocess=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),cat),('num',StandardScaler(),num)])
print('类别特征:',len(cat),'数值特征:',len(num)); print('类别示例:',cat[:6]); print('数值示例:',num[:6])


## 6. 比较逻辑回归与随机森林

两个模型使用相同数据和预处理，在验证集PR-AUC上进行公平比较。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
models={'逻辑回归':Pipeline([('prep',preprocess),('model',LogisticRegression(max_iter=700,class_weight='balanced'))]),'随机森林':Pipeline([('prep',preprocess),('model',RandomForestClassifier(n_estimators=180,min_samples_leaf=10,class_weight='balanced',n_jobs=-1,random_state=108))])}
rows=[]
for name,model in models.items():
    model.fit(X_train,y_train); rows.append([name,average_precision_score(y_val,model.predict_proba(X_val)[:,1])])
validation=pd.DataFrame(rows,columns=['model','validation_PR_AUC']).sort_values('validation_PR_AUC',ascending=False); display(validation.round(3))
best_name=validation.iloc[0].model; best_model=models[best_name].fit(X_dev,y_dev); probability=best_model.predict_proba(X_test)[:,1]


## 7. 测试集概率指标评价

类别不平衡时以PR-AUC为主，同时报告ROC-AUC和概率损失。


In [ ]:
from sklearn.metrics import roc_auc_score,log_loss
metrics=pd.Series({'ROC_AUC':roc_auc_score(y_test,probability),'PR_AUC':average_precision_score(y_test,probability),'LogLoss':log_loss(y_test,probability),'正类率':y_test.mean()})
print('最佳模型:',best_name); print(metrics.round(3).to_string())


## 8. 比较Top-K阈值、Lift与覆盖率

Top-K用于教学性阈值评价，展示精确率、召回率和Lift之间的权衡。


In [ ]:
from sklearn.metrics import confusion_matrix
ranked=pd.DataFrame({'row_id':X_test.index,'actual':y_test.to_numpy(),'probability':probability}).sort_values('probability',ascending=False); threshold_rows=[]
for share in [.05,.10,.20]:
    n=max(1,int(len(ranked)*share)); top=ranked.head(n); threshold_rows.append([f'{share:.0%}',top.probability.min(),top.actual.mean(),top.actual.sum()/ranked.actual.sum(),top.actual.mean()/ranked.actual.mean()])
threshold_table=pd.DataFrame(threshold_rows,columns=['Top比例','概率阈值','Precision','Recall','Lift']); display(threshold_table.round(3))
threshold=threshold_table.loc[threshold_table['Top比例']=='10%','概率阈值'].iloc[0]; prediction=probability>=threshold; print('Top10%混淆矩阵:',confusion_matrix(y_test,prediction).tolist())


## 9. 错误类型与客户分组

区分漏判响应与误报响应，并比较年龄段中的实际率、平均评分和错误率。


In [ ]:
error_df=X_test[['age','job','contact','campaign']].copy(); error_df['actual']=y_test; error_df['probability']=probability; error_df['prediction']=prediction
error_df['error_type']=np.select([(error_df.actual==1)&(~error_df.prediction),(error_df.actual==0)&error_df.prediction],['漏判响应','误报响应'],default='判断正确')
error_df['age_group']=pd.cut(error_df.age,[0,30,45,60,120],labels=['<=30','31-45','46-60','60+'])
age_report=error_df.groupby('age_group',observed=True).agg(customers=('actual','size'),actual_rate=('actual','mean'),mean_score=('probability','mean'),error_rate=('error_type',lambda x:(x!='判断正确').mean()))
print(error_df.error_type.value_counts()); display(age_report.round(3))


## 10. 特征解释与模型局限

置换重要性说明模型主要利用哪些历史信号，同时强调响应预测不等于干预效果预测。


In [ ]:
from sklearn.inspection import permutation_importance
sample_n=min(3000,len(X_test)); sample_idx=np.linspace(0,len(X_test)-1,sample_n,dtype=int)
permutation=permutation_importance(best_model,X_test.iloc[sample_idx],y_test.iloc[sample_idx],n_repeats=3,scoring='average_precision',random_state=108,n_jobs=-1)
importance=pd.Series(permutation.importances_mean,index=features).sort_values(ascending=False)
print('置换重要性前10:\n',importance.head(10).round(4)); print('局限: 数据来自历史营销活动，unknown较多且存在选择机制；响应概率不能解释电话带来的个体因果增量。')


## 结论与表达

- 预测时点决定duration为什么必须排除
- 分层划分保证类别不平衡下的可比性
- PR-AUC和Top-K指标比准确率更有信息
- 响应预测模型不等于因果增量模型


## 项目验收清单

- 完成重复、unknown和类别比例审计
- 排除duration及目标字段
- 完成分层三级划分和Dummy基线
- 比较两个Pipeline模型
- 报告PR-AUC、Top-K、Lift与混淆矩阵
- 完成错误分组与置换重要性分析

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 UCI Bank Marketing 公开数据，按照二分类教学流程建立客户响应预测模型，重点学习事后泄漏、类别不平衡和阈值评价。


### 你已经完成

- 审计重复、unknown和正类比例
- 识别并排除duration事后泄漏
- 使用分层训练验证测试划分
- 建立混合类型预处理Pipeline
- 比较Dummy、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片和特征重要性理解模型


### 建模流程速查

| 阶段 | 学习内容 |
| --- | --- |
| 步骤 1 | 明确通话前预测时点与目标 |
| 步骤 2 | 审计数据质量和类别不平衡 |
| 步骤 3 | 清理重复并探索响应差异 |
| 步骤 4 | 识别duration等禁止字段 |
| 步骤 5 | 分层划分训练、验证和测试 |
| 步骤 6 | 建立混合类型预处理Pipeline |
| 步骤 7 | 比较Dummy、逻辑回归和随机森林 |
| 步骤 8 | 评价PR-AUC、阈值、Lift与覆盖率 |
| 步骤 9 | 分析错误类型和客户分组 |
| 步骤 10 | 解释特征重要性并总结模型局限 |


### 质量与结论提醒

- 分号分隔及字段类型
- unknown的数量和含义
- 重复记录与campaign长尾
- 预测时点决定duration为什么必须排除
- 分层划分保证类别不平衡下的可比性
- PR-AUC和Top-K指标比准确率更有信息
- 响应预测模型不等于因果增量模型


### 学习检查

- [ ] 完成重复、unknown和类别比例审计
- [ ] 排除duration及目标字段
- [ ] 完成分层三级划分和Dummy基线
- [ ] 比较两个Pipeline模型
- [ ] 报告PR-AUC、Top-K、Lift与混淆矩阵
- [ ] 完成错误分组与置换重要性分析


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
